# 01 — Vollständiger Transfer: Global-6 → Sensor 7

Das Notebook startet den vollständigen Workflow aus `calibration_transfer_sensor0.py`. Von jedem der sieben Geräte wird ausschließlich Subsensor 0 verwendet. Die sechs Quellgeräte dienen abwechselnd als LOSO-Pseudoziel, damit die Hyperparameter aller Transfermethoden ohne Zugriff auf Sensor 7 gewählt werden. Verglichen werden Global-6, Global+Target, Scratch, Head-only, Fine-tune, DS und PDS.

## Versuchslogik

1. Sechs LOSO-Folds: fünf Quellgeräte trainieren das Fold-Globalmodell, das sechste ist Pseudoziel.
2. Für jedes Budget werden Lernrate/Epochen von Global+Target, Scratch, Head-only und Fine-tune sowie Alpha/Fenster von DS/PDS per mittlerem LOSO-UGM-RMSE gewählt.
3. Erst nach dieser Auswahl wird das finale Global-6-Modell trainiert.
4. Die eingefrorenen Einstellungen werden auf Sensor 7 angewandt. Sensor 7 beeinflusst die Auswahl nicht.
5. Harte Formprüfung: Jeder Zyklus hat `(1, 1440, 1)`; Sensoren werden niemals als Eingangskanäle gestapelt.

In [ ]:
from pathlib import Path
import json, subprocess, sys
from IPython.display import Image, display

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'Networks' / 'TCOCNNv3.py').exists())
SCRIPT = ROOT / 'Evaluation Seminar' / 'Day_04' / 'calibration_transfer_sensor0.py'
OUTPUT = ROOT / 'artifacts' / 'seminar_day4_global6_sensor0'

# Vollständige Standardeinstellung. Hier können bei Bedarf CLI-Argumente ergänzt werden.
EXTRA_ARGS = []
# Beispiel für einen sehr kurzen technischen Smoke-Test:
# EXTRA_ARGS = ['--budgets','5','--source-epochs','1','--candidate-epochs','1',
#               '--lr-global-target','0.001','--lr-scratch','0.001',
#               '--lr-head','0.0001','--lr-finetune','0.00001',
#               '--alphas','0.1','--windows','5','--loso-folds','1']

command = [sys.executable, '-u', str(SCRIPT), '--output', str(OUTPUT), *EXTRA_ARGS]
print(' '.join(command))

## LOSO-Suche und finaler Transfer

Der vollständige Lauf ist rechenintensiv, weil auch die neuronalen Transfermethoden in jedem LOSO-Fold neu trainiert werden. Die Ausgabe zeigt zuerst sämtliche Entwicklungsfolds und danach die finale Anwendung auf Sensor 7.

In [ ]:
subprocess.run(command, cwd=ROOT, check=True)

## Audit der Auswahl

Die folgenden Dateien dokumentieren die sechs LOSO-Folds, die daraus gewählten Einstellungen und bestätigen, dass Sensor 7 nicht zur Auswahl verwendet wurde.

In [ ]:
config = json.loads((OUTPUT / 'run_config.json').read_text(encoding='utf-8'))
selected = json.loads((OUTPUT / 'selected_loso_settings.json').read_text(encoding='utf-8'))
print(json.dumps(config, indent=2))
print(f'Gewählte Kombinationen: {len(selected)}')
selected[:12]

## Lernkurven und Resultate

Gezeigt werden die Train/Validierungs-Lernkurve des finalen Global-6-Modells, die LOSO-Auswahlkurven und der Vergleich aller Methoden auf Sensor 7.

In [ ]:
for filename in ['source_training_curve.png', 'loso_selected_settings.png',
                 'all_methods_budget_curve.png']:
    print(filename)
    display(Image(filename=str(OUTPUT / filename)))

In [ ]:
metrics_rows = json.loads((OUTPUT / 'final_metrics.json').read_text(encoding='utf-8'))
try:
    import pandas as pd
    display(pd.DataFrame(metrics_rows))
except ImportError:
    print(json.dumps(metrics_rows, indent=2))